# A Gentle Introduction to Pydantic AI

This notebook is a hands-on tutorial that introduces **Pydantic AI**, a Python framework for building LLM-powered applications, and uses it as a vehicle to explore the core ideas behind **Agentic AI**.

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain what Pydantic AI is and why we use a framework instead of calling an LLM API directly.
2. Send single-turn and multi-turn requests to an LLM using `Agent`.
3. Get **structured, type-safe** outputs from an LLM using Pydantic models.
4. Define what an **AI agent** is and identify the role of **tools**.
5. Give an LLM tools (functions it can call) and group them into **toolsets**.
6. Pass runtime information into tools and prompts using **dependency injection**.
7. Use Pydantic AI's **built-in tools** (`WebSearchTool`, `CodeExecutionTool`).

## Prerequisites

- Python 3.12+
- Familiarity with `async`/`await` and basic Pydantic models.
- An Azure OpenAI deployment (or any provider supported by Pydantic AI). Credentials live in `Docker.env`.

## What is Pydantic AI?

Pydantic AI is an "agent framework" — it sits on top of LLM provider SDKs (OpenAI, Anthropic, Azure, Gemini, …) and gives us:

- A single, uniform `Agent` API across providers.
- **Structured outputs**: the LLM returns validated Pydantic models, not just strings.
- **Tool calling**: plain Python functions can be exposed to the model.
- **Dependency injection**: a clean way to pass DB connections, user info, or other runtime context to tools.
- Production niceties: retries, timeouts, message history, streaming, observability.

If you've used LangChain or LlamaIndex, Pydantic AI is in the same family but is intentionally smaller, type-driven, and feels much closer to writing normal Python.

## Notebook Roadmap

1. Setup: env vars and the LLM provider
2. Single-turn and multi-turn LLM calls
3. Structured output with Pydantic models
4. From chatbot to **agent**: giving the LLM tools
5. Dependency injection and runtime context
6. Toolsets, timeouts, retries
7. Built-in tools: web search and code execution
8. Summary and exercises

## 1. Load Environment Variables

We never hardcode API keys in source code. Instead, we keep them in a `.env` file and load them with `python-dotenv`. The file `Docker.env` is expected to define:

- `LLM__AZURE_OPENAI_ENDPOINT`
- `LLM__AZURE_OPENAI_API_KEY`
- `LLM__AZURE_OPENAI_API_VERSION`
- `LLM__MODEL` (e.g. the deployment name of your chat model)
- Optional: `LLM__TEMPERATURE`, `LLM__MAX_TOKENS`

In [48]:
from dotenv import load_dotenv

load_dotenv("Docker.env")

True

## 2. Define the LLM Provider and Model

In Pydantic AI a **provider** describes *where* requests are sent (Azure, OpenAI, Anthropic, …) and a **model** describes *which* model to use through that provider.

We also create a `ModelSettings` object that controls generation behaviour:

| Setting       | Effect                                                          |
| ------------- | --------------------------------------------------------------- |
| `temperature` | Higher = more random/creative; lower = more deterministic.       |
| `max_tokens`  | Hard cap on the length of the response.                          |

We will reuse this `model` object for every example in the notebook.

In [52]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.settings import ModelSettings
import os

# 1. Wadah Provider untuk Groq
groq_provider = OpenAIProvider(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

# 2. GANTI NAMA MODEL ke versi yang aktif saat ini: llama-3.3-70b-versatile
model = OpenAIChatModel(
    "llama-3.3-70b-versatile",
    provider=groq_provider
)

# Pengaturan model tetap bawaan tugasmu
model_settings = ModelSettings(
    temperature=float(os.getenv("LLM__TEMPERATURE", "0.1")),
    max_tokens=int(os.getenv("LLM__MAX_TOKENS", "1000")),
)

## 3. A Simple Conversation with the LLM

The simplest thing we can do is send one prompt and get one answer. In Pydantic AI we always go through the `Agent` class — even for non-agentic, single-turn calls.

Two important parameters:

- **`instructions`**: a system-level message describing the assistant's persona/behavior. It is re-applied fresh on every `agent.run()` call.
- **`user_prompt`**: the actual user message, passed positionally to `agent.run()`.

> **`instructions` vs `system_prompt`**: Pydantic AI also supports `system_prompt`, but it is *carried across* in the message history. For most cases the docs recommend `instructions`, because they are guaranteed to be present at every run.

`agent.run(...)` is asynchronous and returns an `AgentRunResult`; the model's text answer is in `response.output`.

In [53]:
from pydantic_ai import Agent

instruction_prompt = (
    "You are a helpful assistant. You always answer with a single concise sentence."
)

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=instruction_prompt,
    retries=5,
)

user_prompt = "what is Python?"
response = await agent.run(user_prompt)
response.output

'Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility in various applications.'

## 4. Message History (Multi-turn Conversations)

LLMs are **stateless** — every call is independent. If we want the model to remember earlier turns of a conversation, we have to feed the previous messages back in ourselves.

Pydantic AI makes this easy:

- `response.all_messages()` → all messages from the run (instructions + previous turns + new turn).
- `response.new_messages()` → only the messages added in the latest run.

We pass them to the next call via `agent.run(prompt, message_history=...)`.

The example below first shows what happens **without** message history (the model is confused), then shows the same conversation **with** message history (the model correctly understands the follow-up question).

In [54]:
from pydantic_ai import Agent

instruction_prompt = (
    "You are a helpful assistant. You always answer with a single concise sentence."
)

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=instruction_prompt,
    retries=5,
)

In [55]:
first_response = await agent.run("What is Python?")
first_response.output

'Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility in various applications.'

In [56]:
followup_no_history = await agent.run("And when was it released?")
followup_no_history.output

"The information you're referring to is not specified, so I'd need more context to provide a release date."

In [57]:
followup_with_history = await agent.run(
    "And when was it released?",
    message_history=first_response.all_messages(),
)
followup_with_history.output

'Python was first released in 1991 by Guido van Rossum, with the first version, Python 0.9.1, being released in February 1991.'

## 5. Structured Output (Pydantic AI's Killer Feature)

Plain text responses are awkward to use in real software — we usually have to parse them with regex or extra prompting, and they break easily.

Pydantic AI lets us pass a `output_type=SomeModel` argument to `Agent`. Behind the scenes it:

1. Converts the Pydantic model into a JSON schema.
2. Tells the LLM to produce JSON that matches that schema (using the provider's "structured output" or "tool calling" mechanism).
3. Validates the response and returns an instance of the model.

If the LLM produces something invalid, Pydantic AI will retry up to `retries` times.

In the example below, the model receives a free-text sentence and returns a fully-typed `Person` object — no manual parsing needed.

In [58]:
from pydantic import BaseModel


class Person(BaseModel):
    name: str
    age: int
    job: str

In [59]:
from pydantic_ai import Agent

instruction_prompt = (
    "You are a helpful assistant. Extract structured information about a person."
)

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=instruction_prompt,
    retries=5,
    output_type=Person,
)

user_prompt = "John is a 32 years old Police Officer"
response = await agent.run(user_prompt)
response.output

Person(name='John', age=32, job='Police Officer')

In [60]:
type(response.output)

__main__.Person

## 6. (Aside) Embedding Models

So far we've used **chat models** that produce text. There is a second family of models — **embedding models** — that turn a piece of text into a fixed-size vector of numbers. Vectors with similar meanings end up close to each other, which is the foundation of semantic search and Retrieval-Augmented Generation (RAG).

Pydantic AI focuses on agentic LLM behavior; for embeddings you would typically:

- Use the provider SDK directly (e.g. `AzureOpenAI().embeddings.create(...)`), or
- Use a dedicated embeddings library (e.g. `sentence-transformers`, `langchain`, or a vector DB client).

The cell below sketches what an OpenAI embedding call looks like, but it is **commented out** — embeddings are out of scope for this introductory tutorial and would require a separate model deployment.

In [61]:
# Sketch only — requires an embedding deployment, not run in this notebook.
#
# from openai import AzureOpenAI
#
# client = AzureOpenAI(
#     azure_endpoint=os.getenv("LLM__AZURE_OPENAI_ENDPOINT"),
#     api_key=os.getenv("LLM__AZURE_OPENAI_API_KEY"),
#     api_version=os.getenv("LLM__AZURE_OPENAI_API_VERSION"),
# )
# vec = client.embeddings.create(
#     model="text-embedding-3-small",
#     input="Hello, world!",
# ).data[0].embedding
# len(vec)  # e.g. 1536

# Part II — Agentic AI with Pydantic AI

Up to now the LLM has been a **text generator**: we send a prompt, it returns text. That is enough for chatbots and summarizers, but it is not enough for many real applications.

## What is an "AI agent"?

An **AI agent** is an LLM that, in addition to generating text, can:

1. **Decide** that it needs more information or that it should perform an action.
2. **Invoke a tool** — a regular function we wrote — to get that information or perform that action.
3. **Read the result** of the tool and continue reasoning.
4. Repeat steps 1–3 until it has a final answer for the user.

The control loop ("model → maybe call tool → feed result back → model → …") is what makes it *agentic*. The model is not just answering — it is *planning* and *acting*.

```
┌────────────┐    prompt + tool results     ┌────────────┐
│            │  ─────────────────────────►  │            │
│   Agent    │                              │    LLM     │
│  (Python)  │  ◄─────────────────────────  │            │
└────────────┘   text answer  OR  tool call └────────────┘
        │                  ▲
        │ if tool call:    │
        ▼                  │ tool result
   ┌──────────┐            │
   │  Tool    │ ───────────┘
   │ (Python) │
   └──────────┘
```

Pydantic AI implements this loop for us — we just have to:

- Define the tools (Python functions),
- Hand them to the `Agent`,
- Call `agent.run(...)` as usual.

### 7.1 Motivating example: the LLM doesn't know what it doesn't know

Before we add tools, let's see why we need them. The cell below asks the model a question that depends on **information the model cannot possibly have** — the user's favorite color. The best a non-agentic LLM can do is admit it doesn't know.

In [62]:
from pydantic_ai import Agent

instruction_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=instruction_prompt,
    retries=5,
)

user_prompt = "what is my favorite color?"
response = await agent.run(user_prompt)
response.output

"I don't have any information about your favorite color. I'm a large language model, I don't have personal interactions or memories, so I don't know your personal preferences. If you'd like to share your favorite color with me, I'd be happy to chat with you about it!"

As expected, the model has to guess or refuse. Let's give it a tool that can look up favorite colors.

### 7.2 Registering tools — Style 1: pass `tools=[...]` at construction

A **tool** is just a normal Python function. Two things matter for the LLM:

- The **function name** and **type-annotated parameters** become the tool's signature.
- The **docstring** becomes the tool's description — this is what the LLM reads to decide whether to call it.

So good docstrings and good type hints are no longer just for humans — they directly affect agent behavior.

In [63]:
def get_favorite_color(name: str) -> str:
    """Get the favorite color of a person"""
    if name == "John":
        return "Blue"
    elif name == "Jane":
        return "Red"
    else:
        return "Unknown"

In [64]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=system_prompt,
    retries=5,
    tools=[get_favorite_color],
)

user_prompt = "Hi I am Jane, what is my favorite color?"
response = await agent.run(user_prompt)
response.output

'Your favorite color is red.'

### 7.3 Registering tools — Style 2: the `@agent.tool_plain` decorator

The same agent can also be built up *after* construction by decorating functions with `@agent.tool_plain`. This is convenient when the tool is defined close to where it's used, or when you build the agent in one place and add tools in another.

There are two decorator variants:

- `@agent.tool_plain` — the function does **not** need a `RunContext` parameter (no dependencies).
- `@agent.tool` — the function receives a `RunContext` as its first parameter (we'll use this in the next section for dependency injection).

**When to prefer which style?**

| Style                              | Best for                                                     |
| ---------------------------------- | ------------------------------------------------------------ |
| `Agent(..., tools=[...])`          | Static, declarative agents; easy to see all tools at a glance. |
| `@agent.tool_plain` / `@agent.tool` | Dynamically attached tools; tools that need closures or DI.    |

In [65]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=system_prompt,
    retries=5,
)

In [66]:
@agent.tool_plain
def get_favorite_color(name: str) -> str:
    """Get the favorite color of a person"""
    if name == "John":
        return "Blue"
    elif name == "Jane":
        return "Red"
    else:
        return "Unknown"

In [67]:
user_prompt = "Hi I am John, what is my favorite color?"
response = await agent.run(user_prompt)
response.output

'Your favorite color is Blue.'

## 8. Dependency Injection and Runtime Context

In a real application a tool often needs **runtime information** that is not part of the user prompt:

- the currently logged-in user,
- a database connection or an HTTP client,
- the secret answer in a game,
- a feature flag, etc.

We don't want to hardcode these as globals. Pydantic AI provides **dependency injection** through the `deps` mechanism:

1. Tell the agent the **type** of its dependencies via `deps_type=...`.
2. Pass the actual dependency object on every call: `agent.run(prompt, deps=...)`.
3. Inside a tool decorated with `@agent.tool` (note: not `tool_plain`), receive a `RunContext[YourDepsType]`. Use `ctx.deps` to access the dependency.

We'll see two flavors:

- **Case 1**: a tool reads from `ctx.deps`.
- **Case 2**: a dynamic system prompt reads from `ctx.deps`, AND a tool uses a real database connection.

### 8.1 Case 1: Injecting context into a tool

A tiny "guess the number" game. The agent's job is to compare the user's guess to the secret number — but the secret number must come from the **caller**, not from the LLM's prompt.

- `deps_type=int` declares that we will pass an `int` as the dependency.
- `output_type=bool` declares that the final answer is a boolean (won / lost).
- The tool receives the secret via `ctx.deps`.

In [68]:
from pydantic_ai import Agent

system_prompt = "You are a roulette assistant. You can process number guesses from users and return if the guess is correct or not."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=system_prompt,
    retries=5,
    deps_type=int,
    output_type=bool,
)

In [69]:
from pydantic_ai import RunContext


@agent.tool
async def guess_roulette_number(ctx: RunContext[int], guessed_number: int) -> str:
    """Compare the user's guessed number against the secret number injected via deps."""
    return "won" if guessed_number == ctx.deps else "lost"

In [70]:
correct_number = 42
user_prompt = "I am guessing the number is eighteen"
response = await agent.run(user_prompt, deps=correct_number)
response.output

False

In [71]:
correct_number = 42
user_prompt = "I am guessing the number is forty two"
response = await agent.run(user_prompt, deps=correct_number)
response.output

True

### 8.2 Case 2: Dynamic system prompts + a real database tool

Here we demonstrate two more advanced features at once:

1. **Dynamic system prompt** via `@agent.system_prompt`. This decorator registers a *function* that returns a string; that string is added to the system prompt at runtime, after reading from `ctx.deps`. We use it to inject the user's name.
2. **A tool that uses an injected database connection.** The tool no longer carries the connection as a global — it receives it through `ctx.deps`.

We'll define a tiny SQLite database first so the example is self-contained.

In [72]:
import sqlite3
from dataclasses import dataclass


def setup_demo_database(path: str = "database.db") -> sqlite3.Connection:
    """Create (idempotently) a small `person` table for the DI demo."""
    conn = sqlite3.connect(path, check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS person (name TEXT, age INTEGER, profession TEXT)"
    )
    cursor.execute("DELETE FROM person")
    cursor.executemany(
        "INSERT INTO person (name, age, profession) VALUES (?, ?, ?)",
        [
            ("Mike", 20, "Plumber"),
            ("Lisa", 30, "Accountant"),
            ("John", 40, "Programmer"),
        ],
    )
    conn.commit()
    return conn


@dataclass
class DBDependency:
    user_name: str
    db_connection: sqlite3.Connection

In [73]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant. Always start every message by calling the user by name."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=system_prompt,
    retries=5,
    deps_type=DBDependency,
)

In [74]:
from pydantic_ai import RunContext


@agent.system_prompt
def add_user_name(ctx: RunContext[DBDependency]) -> str:
    return f"The user name is {ctx.deps.user_name}."


@agent.tool
def list_all_employees(ctx: RunContext[DBDependency]) -> list[tuple[str, int, str]]:
    """List every employee stored in the database as (name, age, profession)."""
    cursor = ctx.deps.db_connection.cursor()
    cursor.execute("SELECT name, age, profession FROM person;")
    return cursor.fetchall()

In [75]:
db_conn = setup_demo_database()
deps = DBDependency(user_name="boedy", db_connection=db_conn)

response = await agent.run("What employees are in the DB?", deps=deps)
print(response.output)

Boedy, the employees in the database are:
1. Mike, 20 years old, Plumber
2. Lisa, 30 years old, Accountant
3. John, 40 years old, Programmer


## 9. Toolsets — Grouping Related Tools

Once you have more than a handful of tools, it becomes useful to **group** them into reusable bundles. A `Toolset` is exactly that: a named collection of tools you can plug into one or many agents.

Benefits:

- **Reusability** — define a `date_time_toolset` once, plug it into any agent that needs date/time awareness.
- **Modularity** — keep related tools together; the agent definition stays clean.
- **Dynamic attachment** — toolsets can also be passed at `agent.run(..., toolsets=[...])` time, so one agent can grow new capabilities per call.

Below we package three small "what time is it?" tools into a single `FunctionToolset`.

In [76]:
from datetime import datetime
from pydantic_ai.toolsets import FunctionToolset


def get_current_date_tool() -> str:
    today = datetime.today()
    return today.strftime("%Y-%m-%d")


def get_current_weekday_tool() -> str:
    today = datetime.today()
    return today.strftime("%A")


def get_current_time_tool() -> str:
    now = datetime.now()
    return now.strftime("%H:%M:%S")


date_time_toolset = FunctionToolset(
    tools=[get_current_date_tool, get_current_weekday_tool, get_current_time_tool],
)


In [77]:
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=system_prompt,
    retries=5,
    toolsets=[date_time_toolset],
)

user_prompt = "What is the date and time?"
response = await agent.run(user_prompt)
# response = await agent.run(user_prompt, toolsets=[date_time_toolset])

response.output

'The current date is 2026-05-19 and the current time is 23:11:39.'

## 10. Tool Timeouts and Retries

Tools call external code — databases, web APIs, LLMs themselves. They **will** sometimes hang, fail, or take too long. Two safety nets:

- `tool_timeout=N` on the `Agent` — if a single tool call takes longer than `N` seconds it is aborted (raises `TimeoutError`).
- `retries=N` on the `Agent` — if a tool raises (or the LLM produces an invalid structured output), Pydantic AI will give the model another chance up to `N` times.

The example below has a tool that deliberately sleeps for 10 seconds. With `tool_timeout=2` it is interrupted, the model sees the failure, and either retries or apologises. Comment the timeout out to see the "happy path" where the tool eventually returns.

In [90]:
import asyncio
from pydantic_ai import Agent

system_prompt = "You are a helpful assistant. If a tool fails, apologise and explain."

agent = Agent(
    model=model,
    model_settings=model_settings,
    instructions=system_prompt,
    tool_timeout=2,
    retries=1,
)


@agent.tool_plain
async def get_favorite_color_tool() -> str:
    """Look up the user's favorite color (deliberately slow for the demo)."""
    await asyncio.sleep(10)
    return "Blue"


result = await agent.run("What is my favorite color?")
print(result.output)


Traceback (most recent call last):
  File "e:\SEMESTER 4\KULIAH\NLP\pertemuan12\pydantic-ai-gentle-introduction-main\.venv\Lib\site-packages\pydantic_ai\tool_manager.py", line 456, in validate_tool_call
    validated_args = await self._run_validate_hooks(call, tool, ctx, allow_partial=False)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\SEMESTER 4\KULIAH\NLP\pertemuan12\pydantic-ai-gentle-introduction-main\.venv\Lib\site-packages\pydantic_ai\tool_manager.py", line 283, in _run_validate_hooks
    validated_args = await cap.on_tool_validate_error(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        ctx, call=call, tool_def=tool_def, args=raw_args, error=e
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "e:\SEMESTER 4\KULIAH\NLP\pertemuan12\pydantic-ai-gentle-introduction-main\.venv\Lib\site-packages\pydantic_ai\capabilities\combined.py", line 379, in on_tool_validate_error
    raise err

> **Mini-exercise:** comment the `tool_timeout=2` argument out, re-run the cell, and observe how the answer (and the elapsed time) changes.

## 11. Built-in Tools: WebSearchTool and CodeExecutionTool

So far, all tools were Python functions we wrote ourselves. Some tools are common enough that they are provided by the LLM provider itself and integrated by Pydantic AI as **built-in tools**:

- **`WebSearchTool`** — lets the model search the live web. Useful for any question about events that happened after the model's training cutoff.
- **`CodeExecutionTool`** — lets the model write and *run* Python in a sandbox. This is a game-changer for any task involving math, statistics, simulation, or data wrangling, because LLMs are notoriously unreliable at arithmetic but execution gives exact answers.

> **Important:** Both built-in tools require `OpenAIResponsesModel` (not `OpenAIChatModel`). The Responses API is the OpenAI/Azure endpoint that natively supports these tools.

Note also that built-in tools are passed via `builtin_tools=[...]`, not `tools=[...]`.

### 11.1 `WebSearchTool` — giving the agent access to the live web

The model decides on its own whether the question needs a web search. If it does, the tool call happens transparently and the result is woven back into the answer (often with citations).

In [93]:
from pydantic_ai import WebSearchTool
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel
import os

responses_model = OpenAIResponsesModel(
    os.getenv("LLM__MODEL", ""),
    provider=provider,
)

system_prompt = "You are a helpful assistant."
agent = Agent(
    model=responses_model,
    model_settings=model_settings,
    instructions=system_prompt,
    builtin_tools=[WebSearchTool()],
)

user_prompt = "What is most popular movie in 2023?"
response = await agent.run(user_prompt)
print(response.output)


NameError: name 'provider' is not defined

### 11.2 `CodeExecutionTool` — letting the agent write and run Python

The example below asks the agent to estimate π by Monte Carlo simulation. This is the kind of task an LLM should never try to "compute in its head" — instead, it generates Python, executes it in the sandbox, and returns the precise result.

After the run we'll also peek at `response.response.builtin_tool_calls` to see exactly what code the model wrote and ran.

In [94]:
from pydantic_ai import CodeExecutionTool
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel
import os

responses_model = OpenAIResponsesModel(
    os.getenv("LLM__MODEL", ""),
    provider=provider,
)

system_prompt = """You are a precise computational assistant.
When a question involves numerical computation, simulation, statistics, or data analysis,
ALWAYS use the CodeExecutionTool for accuracy instead of approximating."""
agent = Agent(
    model=responses_model,
    model_settings=model_settings,
    instructions=system_prompt,
    builtin_tools=[CodeExecutionTool()],
)

user_prompt = """Run a Monte Carlo simulation to estimate π with at least 200,000 trials.

Instructions for the code:
- Generate random (x, y) points where both x and y are uniformly distributed between -1 and 1.
- Count how many points satisfy x² + y² ≤ 1 (inside the unit circle).
- Estimate π as 4 * (points_inside / total_points).
- Also compute the absolute error compared to the true value of math.pi.
- Measure and report the approximate execution time.

Output a clean report with:
- Estimated π (to 8 decimal places)
- Absolute error
- Number of trials
- Execution time
- A brief explanation of the result's accuracy."""
response = await agent.run(user_prompt)
print(response.output)


NameError: name 'provider' is not defined

In [95]:
for part in response.response.builtin_tool_calls:
    if hasattr(part, "args") and isinstance(part.args, dict) and "code" in part.args:
        print("=== Code the agent ran ===")
        print(part.args["code"])
        print()
    else:
        print("=== Tool result ===")
        print(part.content if hasattr(part, "content") else part)
        print()

C:\Users\JohnWickKW\AppData\Local\Temp\ipykernel_24572\3164962406.py:1: PydanticAIDeprecationWarning: `ModelResponse.builtin_tool_calls` is deprecated, use `ModelResponse.native_tool_calls` instead.
  for part in response.response.builtin_tool_calls:


## 12. Summary

What we learned, in one page:

| Concept                 | API                                                | Why it matters                                     |
| ----------------------- | -------------------------------------------------- | -------------------------------------------------- |
| Provider + model        | `AzureProvider`, `OpenAIChatModel`                 | Decouple "where" from "which" model                |
| Single call             | `Agent(...).run(prompt)`                           | The basic unit of LLM interaction                  |
| Multi-turn              | `agent.run(prompt, message_history=...)`           | LLMs are stateless; we must replay messages        |
| Structured output       | `output_type=MyPydanticModel`                      | Type-safe, validated, no manual parsing            |
| Tool                    | `tools=[fn]` or `@agent.tool_plain`                | Lets the LLM act, not just talk                    |
| Dependency injection    | `deps_type=...` + `RunContext`                     | Pass DBs, users, secrets cleanly                   |
| Dynamic system prompt   | `@agent.system_prompt`                             | Per-call personalization                           |
| Toolset                 | `FunctionToolset`                                  | Reusable bundles of related tools                  |
| Timeout / retries       | `tool_timeout=...`, `retries=...`                  | Defensive programming for flaky tools              |
| Built-in tools          | `WebSearchTool`, `CodeExecutionTool`               | Fresh information + exact computation              |

### The big picture

The reason **agentic AI** is so powerful is not that LLMs got smarter, but that we stopped treating them as pure text generators. We give them a *toolbox* and a *control loop*, and now the same model can:

- Look up live information (web search),
- Compute exact answers (code execution),
- Query our private data (DB tools with DI),
- Take actions in our system (any tool we choose to expose).

Pydantic AI is the framework that makes wiring all of this up feel like writing normal, typed Python.

## 13. Exercises

Try these in the same notebook (or a new one) to consolidate:

1. **Calculator agent.** Build an agent with two tools, `add(a: int, b: int)` and `multiply(a: int, b: int)`. Ask it: *"What is (3 + 4) * 5?"* and verify it calls both tools.
2. **Structured + agent.** Modify the SQLite example so it returns a `list[Employee]` (a Pydantic model) instead of free-form text.
3. **Conversation memory.** Wrap the chat agent in a small Python class that automatically forwards `message_history` between turns. Test it with a 3-turn conversation.
4. **Tool error handling.** Add a tool that occasionally raises `ValueError`. Set `retries=3` and observe what the agent does. Then remove `retries` and observe again.
5. **Pick the right built-in tool.** Ask one agent (with both `WebSearchTool` and `CodeExecutionTool`): *"What was the average closing price of AAPL last week?"*. Inspect `builtin_tool_calls` to see whether the model used search, code, or both.

## 14. Further Reading

- Pydantic AI docs — [https://ai.pydantic.dev](https://ai.pydantic.dev)
- "Building Effective Agents" by Anthropic — design patterns for agent loops.
- OpenAI Responses API guide — context for `WebSearchTool` and `CodeExecutionTool`.
- The original Pydantic library — [https://docs.pydantic.dev](https://docs.pydantic.dev) (the data-validation engine that everything here is built on).